# E2E Type Progression

This notebook illustrates the typed interface progression used by `mellea-lrc`:

1. `Path` -> `PreprocessedDocument`
2. `PreprocessedDocument` -> `DocumentExtraction`
3. `DocumentExtraction` -> `DocumentValidation`
4. `DocumentValidation` -> `DocumentAssessment`

Each step writes `local/snapshots/<doc>/<stage>.json`, reloads through `deserialize_*`, and reads its input from the previous snapshot. Re-run any step after setup as long as upstream snapshots exist.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path


from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

from mellea_lrc.assessment import (
    CitationAssessment,
    MelleaCallContext,
    run_assessment_async,
)
from mellea_lrc.extraction import run_extraction
from mellea_lrc.llm import llm_provider_config_from_env
from mellea_lrc.preprocessing import preprocess
from mellea_lrc.serialization import (
    deserialize_document_assessment,
    deserialize_document_extraction,
    deserialize_document_validation,
    deserialize_preprocessed_document,
    serialize_document_assessment,
    serialize_document_extraction,
    serialize_document_validation,
    serialize_preprocessed_document,
)
from mellea_lrc.validation import run_validation

TEST_DATA_DIR = Path("../local/test_data")
SNAPSHOT_ROOT = Path("../local/snapshots")
DOC_PATH_LIST = sorted(TEST_DATA_DIR.glob("*.txt"))

if not DOC_PATH_LIST:
    raise FileNotFoundError(f"No .txt files found in {TEST_DATA_DIR.resolve()}")

from collections.abc import Callable

SNAPSHOT_STAGES = ("preprocessed", "extraction", "validation", "assessment")


def snapshot_path(stage: str) -> Path:
    if stage not in SNAPSHOT_STAGES:
        msg = f"Unknown snapshot stage: {stage}"
        raise ValueError(msg)
    return SNAPSHOT_DIR / f"{stage}.json"


def write_snapshot(stage: str, payload: object) -> Path:
    path = snapshot_path(stage)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def load_snapshot(stage: str, deserialize: Callable[[dict[str, object]], object]) -> object:
    path = snapshot_path(stage)
    if not path.exists():
        msg = f"Missing snapshot: {path}"
        raise FileNotFoundError(msg)
    payload = json.loads(path.read_text(encoding="utf-8"))
    return deserialize(payload)


def summarize_snapshot(stage: str, path: Path, payload: dict[str, object]) -> dict[str, object]:
    summary = {
        "stage": stage,
        "snapshot": str(path),
        "top_level_keys": sorted(payload.keys()),
    }
    counts = payload.get("counts")
    if isinstance(counts, dict):
        summary["counts"] = counts
    return summary

In [ ]:
DOC_PATH = DOC_PATH_LIST[0]
SNAPSHOT_DIR = SNAPSHOT_ROOT / DOC_PATH.stem
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

display(
    {
        "doc_path": str(DOC_PATH),
        "snapshot_dir": str(SNAPSHOT_DIR),
    }
)

## Step 1: Preprocess

`preprocess(Path)` produces the first typed boundary object: `PreprocessedDocument`. The snapshot is reloaded with `deserialize_preprocessed_document` before extraction.

In [ ]:
preprocessed_raw = preprocess(DOC_PATH)
preprocessed_payload = serialize_preprocessed_document(preprocessed_raw)
preprocessed_path = write_snapshot("preprocessed", preprocessed_payload)
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)

display(type(preprocessed))
display(
    {
        **summarize_snapshot("preprocessed", preprocessed_path, preprocessed_payload),
        "chars": len(preprocessed.text),
        "source_path": preprocessed.metadata.source_path,
        "backend": preprocessed.metadata.backend.value,
    }
)

## Step 2: Extract

`run_extraction(PreprocessedDocument)` advances the object to `DocumentExtraction`. The snapshot is reloaded with `deserialize_document_extraction` before validation.

In [ ]:
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)
extraction_raw = run_extraction(preprocessed)
extraction_payload = serialize_document_extraction(extraction_raw)
extraction_path = write_snapshot("extraction", extraction_payload)
extraction = load_snapshot("extraction", deserialize_document_extraction)

display(type(extraction))
display(summarize_snapshot("extraction", extraction_path, extraction_payload))
display(extraction_payload["citations"][:3])

## Step 3: Validate

`run_validation(DocumentExtraction)` advances the object to `DocumentValidation`. The snapshot is reloaded with `deserialize_document_validation` before assessment.

This may call the configured CourtListener access service.

In [ ]:
extraction = load_snapshot("extraction", deserialize_document_extraction)
validation_raw = run_validation(extraction)
validation_payload = serialize_document_validation(validation_raw)
validation_path = write_snapshot("validation", validation_payload)
validation = load_snapshot("validation", deserialize_document_validation)

display(type(validation))
display(summarize_snapshot("validation", validation_path, validation_payload))
display(validation_payload["validations"][:3])

## Step 4: Assess

`run_assessment_async(DocumentValidation)` advances the object to `DocumentAssessment` and may call the configured LLM provider in parallel. The snapshot is reloaded with `deserialize_document_assessment`.

In [ ]:
import time

validation = load_snapshot("validation", deserialize_document_validation)
provider_config = llm_provider_config_from_env(os.environ)
mellea_started: dict[str, float] = {}


def on_mellea_call(ctx: MelleaCallContext) -> None:
    mellea_started[ctx.citation_id] = time.perf_counter()
    print(
        f"[mellea {ctx.mellea_call}] start id={ctx.citation_id} "
        f"extracted={ctx.extracted_case_name!r}",
        flush=True,
    )


def on_mellea_done(ctx: MelleaCallContext, item: CitationAssessment) -> None:
    elapsed = time.perf_counter() - mellea_started[ctx.citation_id]
    case = item.case_assess.status.value
    print(
        f"[mellea {ctx.mellea_call}] done  id={ctx.citation_id} "
        f"case={case} ({elapsed:.1f}s)",
        flush=True,
    )


assessment_raw = await run_assessment_async(
    validation,
    mellea_concurrency=5,
    on_mellea_call=on_mellea_call,
    on_mellea_done=on_mellea_done,
)
assessment_payload = serialize_document_assessment(assessment_raw)
assessment_path = write_snapshot("assessment", assessment_payload)
assessment = load_snapshot("assessment", deserialize_document_assessment)

display(
    {
        "assessment_path": str(assessment_path),
        "assessments": len(assessment.assessments),
        "modified": len(assessment.modified_citations),
        "reassessments": len(assessment.reassessments),
    }
)

display(type(assessment))
display(
    {
        **summarize_snapshot("assessment", assessment_path, assessment_payload),
        "provider": provider_config.provider.value,
        "model": provider_config.model,
        "assessments": len(assessment.assessments),
        "modified_citations": len(assessment.modified_citations),
        "reassessments": len(assessment.reassessments),
    }
)